# Libraries

In [7]:
!pip install gensim plotly scikit-learn


In [8]:
import os
import multiprocessing
import pandas as pd
import numpy as np
import ast
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import plotly.express as px


## Environment setup

In your Google Drive, create a folder "CLT" and upload the csv with its original name (downloaded from Kaggle).

Switch: "local" (VS Code + GitHub) | "colab" (Google Colab + Drive)

In [9]:
ENV = "colab"

In [10]:
if ENV == "colab":
    # from google.colab import userdata
    from google.colab import drive
    drive.mount('/content/drive')
    # Load Gemini API key from Colab Secrets (set via the key icon in the sidebar)
    # GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


> **Note:** LangExtract requires a Gemini API key. You can obtain one for free at [aistudio.google.com](https://aistudio.google.com/app/apikey) and store it securely as a Colab Secret.

# Stage 2 Dataset Preparation and Export


## 1.1 Introduction

At this point Stage 1 preprocessing pipeline is complete.
Dataset after Stage 1 is about 1.5GB and we decided to
save only necessary data to `ai_media_norm.csv` containing only the 8 columns needed for Stage 2, discarding all intermediate token and lemma variants that were used internally during cleaning.
This also avoids repeating the computationally expensive and long spaCy processing (tokenization, lemmatization).

**Saved columns:** `date`, `domain`, `url`, `tags`, `title`, `content`, `title_lemma_norm`, `content_lemma_norm`.

This keeps the file small and focused — Stage 2 only needs the original metadata, the cleaned readable text, and the final normalized lemma lists for embedding training.

Following commented code should show what was done at the end of the preprocessing step in Stage 1.



In [11]:
# This code doesnt need to be run, its just a demonstration of how to save the normalized data for Stage 2.

# Save only the columns needed for Stage 2 as a checkpoint
# stage2_input_cols = [
#     'date', 'domain', 'url', 'tags',
#     'title', 'content',
#     'title_lemma_norm', 'content_lemma_norm',
# ]

# if ENV == "colab":
#     norm_path = '/content/drive/My Drive/CLT/stage_2/ai_media_norm.csv'
# else:
#     norm_path = 'data/stage_2/ai_media_norm.csv'

# os.makedirs(os.path.dirname(norm_path), exist_ok=True)
# df_norm[stage2_input_cols].to_csv(norm_path, index=False, encoding='utf-8')

# print("Saved to:", norm_path)
# print("Shape:", df_norm[stage2_input_cols].shape)

## 1.2 Load Stage 1 Checkpoint

We load the preprocessed dataset (`ai_media_norm.csv`)

Since CSV does not preserve Python list types, we use `ast.literal_eval` to restore `title_lemma_norm`, `content_lemma_norm`, and `tags` back to proper Python lists. The `date` column is also re-parsed to `datetime`.


In [12]:
# Load the Stage 1 checkpoint
if ENV == "colab":
    norm_path = '/content/drive/My Drive/CLT/ai_media_norm.csv'
else:
    norm_path = 'data/ai_media_norm.csv'

df_norm_loaded = pd.read_csv(norm_path, encoding='utf-8')

# Restore list columns serialized as strings by CSV
for col in ['title_lemma_norm', 'content_lemma_norm', 'tags']:
    df_norm_loaded[col] = df_norm_loaded[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

df_norm_loaded['date'] = pd.to_datetime(df_norm_loaded['date'])

print("Loaded shape:", df_norm_loaded.shape)


Loaded shape: (16518, 8)


## 1.3 Build Stage 2 Columns

- **`doc_id`**: Stable zero-padded identifier (e.g. `doc_00042`) — survives CSV export and links model outputs back to source articles.
- **`text_clean_light`**: Readable text from cleaned `title` + `content` — used for display, search, and labeling. Stop words kept, natural sentence structure preserved.
- **`text_for_embedding`**: Input for Word2Vec / FastText / Doc2Vec — `title_lemma_norm` + `content_lemma_norm` joined as a space-separated string. Stop words removed, noise filtered.
- **`token_count`**: Number of tokens in `text_for_embedding`, used for quality filtering in the next step.
- **`split`**: Reproducible 80/20 train/val assignment with fixed random seed (42).


In [13]:
df_stage2 = df_norm_loaded.copy().reset_index(drop=True)

# Unique document ID
df_stage2['doc_id'] = ['doc_' + str(i).zfill(5) for i in range(len(df_stage2))]

# Readable combined text: cleaned title + cleaned content
df_stage2['text_clean_light'] = (
    df_stage2['title'].fillna('') + '. ' + df_stage2['content'].fillna('')
).str.strip('. ')

# Embedding text: final normalized lemma tokens from title + content joined
df_stage2['text_for_embedding'] = df_stage2.apply(
    lambda row: ' '.join(
        (row['title_lemma_norm'] if isinstance(row['title_lemma_norm'], list) else []) +
        (row['content_lemma_norm'] if isinstance(row['content_lemma_norm'], list) else [])
    ),
    axis=1
)

# Token count
df_stage2['token_count'] = df_stage2['text_for_embedding'].str.split().str.len()

# Reproducible 80/20 train/val split
np.random.seed(42)
split_mask = np.random.rand(len(df_stage2)) < 0.8
df_stage2['split'] = np.where(split_mask, 'train', 'val')


## 1.4 Filter, Export and Verify

Before exporting we apply three quality filters to ensure only meaningful documents reach the embedding model:

1. **Remove empty embedding texts** — rows where `text_for_embedding` is blank or whitespace-only are dropped. These would produce zero-length inputs that crash or corrupt model training.
2. **Remove very short texts** — rows with fewer than 5 tokens are dropped. Documents this short carry almost no semantic signal and can distort word co-occurrence statistics in Word2Vec / FastText.
3. **Drop embedding duplicates** — rows whose `text_for_embedding` is identical to another row are deduplicated. Repeated documents would over-weight certain terms in the trained embeddings.

The index is then reset to close any gaps left by the filters, ensuring clean positional access. The final dataframe is narrowed to the 11 export columns and written to three CSV files — `ai_media_stage2_full.csv`, `ai_media_stage2_train.csv`, and `ai_media_stage2_val.csv` (all UTF-8). A summary is printed showing the final shape, train/val counts, and a sample of the key columns.


In [14]:
# Remove empty / very short embedding texts
df_stage2 = df_stage2[df_stage2['text_for_embedding'].str.strip() != '']
df_stage2 = df_stage2[df_stage2['token_count'] >= 5]

# Drop duplicates based on embedding text
df_stage2 = df_stage2.drop_duplicates(subset='text_for_embedding').reset_index(drop=True)

# Final column selection
export_cols = ['doc_id', 'date', 'domain', 'url', 'tags',
               'title', 'content', 'text_clean_light',
               'text_for_embedding', 'token_count', 'split']
df_export = df_stage2[export_cols]

if ENV == "colab":
    base_path = '/content/drive/My Drive/CLT/stage_2/'
else:
    base_path = 'data/stage_2/'

os.makedirs(base_path, exist_ok=True)

# Full dataset
df_export.to_csv(base_path + 'ai_media_stage2_full.csv', index=False, encoding='utf-8')

# Train / val splits — ready for downstream supervised tasks
df_export[df_export['split'] == 'train'].to_csv(base_path + 'ai_media_stage2_train.csv', index=False, encoding='utf-8')
df_export[df_export['split'] == 'val'].to_csv(base_path + 'ai_media_stage2_val.csv', index=False, encoding='utf-8')

# Summary
print("Export shape:", df_export.shape)
print("\nSplit counts:")
print(df_export['split'].value_counts())
print("\nSample (first 3 rows):")
sample = df_export[['doc_id', 'text_clean_light', 'text_for_embedding', 'token_count', 'split']].head(3).copy()
sample['text_clean_light'] = sample['text_clean_light'].str[:80] + '...'
sample['text_for_embedding'] = sample['text_for_embedding'].str[:80] + '...'
print(sample.to_string())

Export shape: (16518, 11)

Split counts:
split
train    13224
val       3294
Name: count, dtype: int64

Sample (first 3 rows):
      doc_id                                                                     text_clean_light                                                                   text_for_embedding  token_count  split
0  doc_00000  ricoh to provide customer support for agility robotics digit humanoid. the digit...  ricoh provide customer support agility robotic digit humanoid digit humanoid wor...          501  train
1  doc_00001  ai design trends to watch in 2025 from photorealism to personalization. artifici...  ai design trend watch photorealism personalization artificial intelligence ai ra...          804    val
2  doc_00002  ais generate more novel and exciting research ideas than human experts. the firs...  ais generate novel exciting research idea human expert statistically significant...          533  train


# 2. Word2Vec Embeddings


We train a **Skip-gram Word2Vec** model on the `text_for_embedding` column of the training split.
The input tokens are the pre-processed lemma sequences produced in Stage 1 (stop words removed, normalised).

**Role in the broader pipeline:**  
Word2Vec operates on individual lemma tokens, not BPE subwords, so its vectors do not plug directly into a TinyLlama-style transformer.
They serve as:
- a **domain vocabulary sanity check** — nearest-neighbour probes reveal whether the corpus signal is coherent;
- **embedding initialisation** for lightweight classification or retrieval heads built on top of the LLM;
- **input features** for non-neural baselines (TF-IDF + W2V centroid classifiers, etc.).

**Hyperparameters chosen:**

| Parameter | Value | Rationale |
|---|---|---|
| `vector_size` | 200 | Good balance for ~16 k documents |
| `window` | 5 | Standard context window |
| `min_count` | 5 | Drop tokens appearing fewer than 5 times |
| `sg` | 1 | Skip-gram; better for rare domain terms |
| `epochs` | 10 | Sufficient for convergence on this corpus size |


## 2.1 Load & Tokenize

We load the training split from `ai_media_stage2_train.csv`.
Each row's `text_for_embedding` is already a space-separated string of lemma tokens — we split it into a Python list to get the `List[List[str]]` format that gensim expects.


In [15]:
if ENV == "colab":
    train_path = '/content/drive/My Drive/CLT/stage_2/ai_media_stage2_train.csv'
else:
    train_path = 'data/stage_2/ai_media_stage2_train.csv'

df_train = pd.read_csv(train_path, encoding='utf-8')

# Split space-separated token strings into lists — required format for gensim
sentences = [text.split() for text in df_train['text_for_embedding'].dropna()]

print(f"Documents  : {len(sentences):,}")
print(f"Total tokens: {sum(len(s) for s in sentences):,}")
print(f"Sample     : {sentences[0][:12]} ...")


Documents  : 13,224
Total tokens: 11,340,664
Sample     : ['ricoh', 'provide', 'customer', 'support', 'agility', 'robotic', 'digit', 'humanoid', 'digit', 'humanoid', 'work', 'distribution'] ...


## 2.2 Train Word2Vec

Training uses all available CPU cores.


In [16]:
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=200,   # embedding dimensions
    window=5,          # context window
    min_count=5,       # ignore tokens appearing fewer than 5 times
    sg=1,              # 1 = Skip-gram, 0 = CBOW
    workers=1,         # single worker avoids gensim Cython GC teardown exceptions
    epochs=10,
    seed=42,
)

print(f"Vocabulary size : {len(w2v_model.wv):,} tokens")
print(f"Vector dimensions: {w2v_model.wv.vector_size}")


Vocabulary size : 42,370 tokens
Vector dimensions: 200


## 2.3 Save Model

Two artefacts are saved to `data/stage_2/embeddings/`:

- **`w2v_ai_media.model`** — full gensim model (can be reloaded for further training or inference).
- **`w2v_ai_media_vectors.txt`** — plain-text word vectors in `word2vec` format, compatible with external tools (FastText, Magnitude, `load_facebook_vectors`, etc.).


In [17]:
if ENV == "colab":
    model_dir = '/content/drive/My Drive/CLT/stage_2/embeddings/'
else:
    model_dir = 'data/stage_2/embeddings/'

os.makedirs(model_dir, exist_ok=True)

# Full gensim model (reloadable)
w2v_model.save(model_dir + 'w2v_ai_media.model')

# Plain-text vectors (word2vec format — compatible with external tools)
w2v_model.wv.save_word2vec_format(model_dir + 'w2v_ai_media_vectors.txt', binary=False)

print("Saved to:", model_dir)


Saved to: /content/drive/My Drive/CLT/stage_2/embeddings/


## 2.4 Evaluate

Nearest-neighbour probes confirm the model has learned domain-coherent representations.
We test a small set of AI-media terms — adjust `probe_words` to match your corpus vocabulary.
We also print the top-20 tokens by frequency to quickly spot any noise that survived the earlier cleaning steps.


In [18]:
wv = w2v_model.wv

# --- Nearest-neighbour probes ---
probe_words = ['ai', 'model', 'work', 'bias', 'security']

print("=== Nearest-neighbour probes (top 5) ===")
for word in probe_words:
    if word in wv:
        neighbours = wv.most_similar(word, topn=5)
        neighbours_str = ', '.join(f"{w} ({s:.2f})" for w, s in neighbours)
        print(f"  {word:<12} → {neighbours_str}")
    else:
        print(f"  {word:<12} → not in vocabulary")

# --- Top-20 tokens by frequency ---
print("\n=== Top-20 tokens by frequency ===")
vocab_counts = sorted(wv.key_to_index.keys(),
                      key=lambda w: w2v_model.wv.get_vecattr(w, 'count'),
                      reverse=True)[:20]
for i, w in enumerate(vocab_counts, 1):
    count = w2v_model.wv.get_vecattr(w, 'count')
    print(f"  {i:>2}. {w:<20} {count:,}")


=== Nearest-neighbour probes (top 5) ===
  ai           → genai (0.67), generative (0.61), matcher (0.56), agentic (0.56), ragop (0.55)
  model        → autorater (0.69), modelsa (0.66), grm (0.66), mllm (0.66), magistral (0.66)
  work         → talente (0.63), collaborate (0.56), fulfilled (0.55), apploi (0.55), costigan (0.55)
  bias         → biased (0.66), fairness (0.66), perpetuate (0.65), discrimination (0.61), prejudice (0.60)
  security     → cybersecurity (0.67), sentra (0.56), protection (0.55), blackpoint (0.54), posture (0.53)

=== Top-20 tokens by frequency ===
   1. ai                   163,854
   2. use                  78,932
   3. datum                74,577
   4. model                72,459
   5. company              65,933
   6. new                  47,417
   7. time                 45,220
   8. like                 44,973
   9. technology           43,465
  10. system               41,884
  11. year                 39,739
  12. business             37,399
  13. wor

**Embedding evaluation**

The Word2Vec model captures meaningful semantic relationships within the AI discourse dataset. For example, the word “bias” is strongly associated with fairness, discrimination, and prejudice, reflecting discussions around AI ethics. Similarly, “ai” clusters with terms such as generative and genai, indicating the prominence of generative AI topics in the corpus.

**Dataset structure**

The most frequent tokens show that the dataset primarily discusses AI technologies and industry adoption. Terms such as model, data, technology, and company dominate the vocabulary, confirming the dataset’s focus on technological developments and business applications of AI

# 3. Cluster Visualisation

We project the Word2Vec **vocabulary** into 2D using **t-SNE**, then assign each word to a thematic cluster using **K-Means**.
Each point in the interactive plot is one vocabulary token. Points that are close together share similar contexts in the corpus — hovering reveals the word and its frequency.

**Pipeline:**
1. Take the top `TOP_N` most-frequent words (keeps the plot readable and focuses on signal-rich terms).
2. L2-normalise the vectors (equivalent to cosine similarity for K-Means).
3. Reduce to 2D with t-SNE (`perplexity=40`, `n_iter=1000`).
4. Cluster with K-Means (`N_CLUSTERS` groups).
5. Render as an interactive Plotly scatter — point size scales with token frequency.


## 3.1 Configure & Extract Vectors

Adjust `TOP_N` and `N_CLUSTERS` to trade off detail vs. readability.
A good starting point is 2 000–4 000 words and 6–10 clusters for an AI-media corpus of this size.


In [19]:
TOP_N      = 3000   # most-frequent words to include in the plot
N_CLUSTERS = 8      # number of thematic clusters

# Descriptive cluster names — update if you re-run with different N_CLUSTERS or seed
CLUSTER_NAMES = {
    0: 'General Narrative',
    1: 'Policy & Governance',
    2: 'Media & Information',
    3: 'Innovation & Development',
    4: 'Research & Society',
    5: 'Industry & Business',
    6: 'AI Models & Data',
    7: 'Systems & Tools',
}

# Sort vocabulary by frequency descending, take top N
words_sorted = sorted(
    wv.key_to_index.keys(),
    key=lambda w: wv.get_vecattr(w, 'count'),
    reverse=True
)[:TOP_N]

vectors = np.array([wv[w] for w in words_sorted])
freqs   = np.array([wv.get_vecattr(w, 'count') for w in words_sorted])

print(f"Words selected : {len(words_sorted):,}")
print(f"Vector matrix  : {vectors.shape}")
print(f"Frequency range: {freqs.min():,} – {freqs.max():,}")


Words selected : 3,000
Vector matrix  : (3000, 200)
Frequency range: 502 – 163,854


## 3.2 t-SNE Reduction & K-Means Clustering

Vectors are L2-normalised before t-SNE so distances approximate cosine similarity.
t-SNE takes ~30–60 s on 3 000 points with `n_jobs=1` (single-threaded to avoid Cython issues).

K-Means is applied **on the 2D t-SNE coordinates** (not the original 200-dim vectors).
This ensures every cluster forms a spatially contiguous region in the plot — no colour will appear in two separate corners.


In [20]:
# L2-normalise (cosine similarity)
vectors_norm = normalize(vectors)

# t-SNE: reduce 200-dim → 2-dim
print("Running t-SNE …")
tsne   = TSNE(n_components=2, perplexity=40, max_iter=1000, random_state=42, n_jobs=1)
coords = tsne.fit_transform(vectors_norm)

# K-Means on 2D t-SNE coords — clusters will be spatially coherent in the plot
km     = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init='auto')
labels = km.fit_predict(coords)

print("Done.\nCluster sizes:")
unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    mask      = labels == u
    top_words = [words_sorted[i] for i in np.where(mask)[0][:5]]
    print(f"  Cluster {u} ({c:>4} words) — sample: {', '.join(top_words)}")


Running t-SNE …
Done.
Cluster sizes:
  Cluster 0 ( 353 words) — sample: need, security, real, result, performance
  Cluster 1 ( 355 words) — sample: technology, high, lead, drive, report
  Cluster 2 ( 340 words) — sample: ai, create, user, agent, human
  Cluster 3 ( 420 words) — sample: time, like, research, world, people
  Cluster 4 ( 425 words) — sample: industry, cost, global, policy, energy
  Cluster 5 ( 317 words) — sample: company, new, year, business, work
  Cluster 6 ( 411 words) — sample: datum, model, system, large, language
  Cluster 7 ( 379 words) — sample: use, help, include, provide, tool


The clustering reveals that the AI discourse in the dataset spans several dimensions, including technology development, industrial adoption, governance and risk, global policy, and research outcomes. The embedding model therefore captures meaningful semantic relationships within the AI media corpus.

**Cluster 0 — General discussion and narrative language.**  
Words: need, lead, good, way, report<br>
This cluster contains general narrative and descriptive terms commonly used across many articles. These words often appear in explanations, summaries, or commentary rather than referring to a specific AI topic.

**Cluster 1 — Global policy, governance, and investment.**  
Words: right, global, organization, policy, investment<br>
This cluster reflects discussions about international policy, governance frameworks, and organizational strategies related to AI development and regulation.

**Cluster 2 — Media content and information access.**  
Words: x80, think, know, content, access<br>
This cluster appears to represent discourse related to information sharing, media content, and knowledge access. Such language is often used when discussing communication, digital platforms, or public engagement with AI technologies.

**Cluster 3 — Innovation, development, and technological change.**  
Words: build, base, high, drive, change<br>
This cluster reflects language associated with technological development and innovation processes, describing how AI systems are built, improved, and used to drive change.

**Cluster 4 — Research, time trends, and societal context.**  
Words: new, time, year, research, people<br>
This cluster represents discussions of AI research developments over time and their broader societal implications, often appearing in reports on recent advancements or research progress.

**Cluster 5 — Industry, companies, and business applications.**  
Words: company, technology, business, work, customer<br>
This cluster reflects the industrial and commercial dimension of AI, focusing on how companies apply AI technologies in products, services, and customer-facing applications.

**Cluster 6 — AI methods, models, and data infrastructure.**  
Words: ai, datum, model, language, user<br>
This cluster captures the technical core of AI discourse, including references to machine learning models, data, language models, and interactions with users.

**Cluster 7 — Systems, tools, and practical AI usage.**  
Words: use, like, system, help, include<br>
This cluster reflects practical applications of AI systems and tools, describing how technologies are used to support tasks, services, and everyday technological workflows.

## 3.3 Interactive Plot

Each point is one vocabulary word.
**Size** scales with token frequency — larger dots are the most common terms in the corpus.
**Colour** represents the K-Means cluster.
Hover any point to see the word and its raw frequency count.


In [21]:

# --- Build top-words lookup (used for fallback if name not set) ---
cluster_top_words = {}
for cid in sorted(np.unique(labels)):
    idx_sorted = sorted(np.where(labels == cid)[0], key=lambda i: freqs[i], reverse=True)
    cluster_top_words[cid] = [words_sorted[i] for i in idx_sorted[:3]]

def _cluster_label(cid):
    return CLUSTER_NAMES.get(cid) or ' / '.join(cluster_top_words[cid])

# Centroid coordinates in t-SNE space
centroids = {
    cid: (coords[labels == cid, 0].mean(), coords[labels == cid, 1].mean())
    for cid in sorted(np.unique(labels))
}

df_viz = pd.DataFrame({
    'x'      : coords[:, 0],
    'y'      : coords[:, 1],
    'word'   : words_sorted,
    'cluster': [_cluster_label(l) for l in labels],
    'freq'   : freqs,
})

fig = px.scatter(
    df_viz,
    x='x', y='y',
    color='cluster',
    hover_name='word',
    hover_data={'freq': ':,', 'cluster': True, 'x': False, 'y': False},
    size='freq',
    size_max=25,
    opacity=0.70,
    title=f't-SNE of Word2Vec vocabulary — {N_CLUSTERS} K-Means clusters  (top {TOP_N:,} words)',
    width=1200,
    height=840,
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_traces(marker=dict(line=dict(width=0)))

# Centroid label boxes — cluster name
for cid, (cx, cy) in centroids.items():
    fig.add_annotation(
        x=cx, y=cy,
        text=f'<b>{_cluster_label(cid)}</b>',
        showarrow=False,
        font=dict(size=12, color='#111'),
        bgcolor='rgba(255,255,255,0.82)',
        bordercolor='#999',
        borderwidth=1,
        borderpad=5,
    )

fig.update_layout(
    legend_title_text='Cluster',
    legend=dict(font=dict(size=11)),
    xaxis=dict(showticklabels=False, title='', showgrid=False, zeroline=False),
    yaxis=dict(showticklabels=False, title='', showgrid=False, zeroline=False),
)
fig.show()


MUSS ANGEPASST WERDEN, LIEFERT ANDERE ANORDNUNG
**Spatial relationships in the plot**

The spatial layout also reveals relationships between topics:

General narrative words (C0) appear near the center because they occur across many contexts.

Technical AI concepts (C6) appear close to system usage (C7), reflecting how models and data connect to real-world applications.

Industry/business topics (C5) are positioned near technology development clusters, suggesting that AI innovation is closely tied to commercial adoption.

